In [2]:
#Algorithms: SVM, Logistic Regression, Random Forest, XGBoost.
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score,f1_score
import matplotlib.pyplot as plt
import seaborn as sns


In [4]:
df=pd.read_csv('diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
# Check for missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [6]:
x=df.drop('Outcome',axis=1)
y=df['Outcome']

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [8]:
#train test split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

# Feature Scaling
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)



In [12]:
#apply algorithms
models = {
    'SVM': SVC(probability=True),
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(),
    'XGBoost': XGBClassifier()
}
results = {}
for name, model in models.items():
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    y_prob = model.predict_proba(x_test)[:, 1]
    results[name] = {
        'classification_report': classification_report(y_test, y_pred, output_dict=True),
        'accuracy': accuracy_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_prob),
        'confusion_matrix': confusion_matrix(y_test, y_pred)
    }



In [13]:
for name, metrics in results.items():
    print(f"{name}: classification_report=\n{metrics['classification_report']}\nAccuracy={metrics['accuracy']:.4f}, F1 Score={metrics['f1_score']:.4f}, Confusion Matrix=\n{metrics['confusion_matrix']}\n, ROC AUC={metrics['roc_auc']:.4f}")

SVM: classification_report=
{'0': {'precision': 0.7735849056603774, 'recall': 0.8282828282828283, 'f1-score': 0.8, 'support': 99.0}, '1': {'precision': 0.6458333333333334, 'recall': 0.5636363636363636, 'f1-score': 0.6019417475728155, 'support': 55.0}, 'accuracy': 0.7337662337662337, 'macro avg': {'precision': 0.7097091194968554, 'recall': 0.6959595959595959, 'f1-score': 0.7009708737864078, 'support': 154.0}, 'weighted avg': {'precision': 0.7279593441150045, 'recall': 0.7337662337662337, 'f1-score': 0.7292649098474341, 'support': 154.0}}
Accuracy=0.7338, F1 Score=0.6019, Confusion Matrix=
[[82 17]
 [24 31]]
, ROC AUC=0.8051
Logistic Regression: classification_report=
{'0': {'precision': 0.8144329896907216, 'recall': 0.797979797979798, 'f1-score': 0.8061224489795918, 'support': 99.0}, '1': {'precision': 0.6491228070175439, 'recall': 0.6727272727272727, 'f1-score': 0.6607142857142857, 'support': 55.0}, 'accuracy': 0.7532467532467533, 'macro avg': {'precision': 0.7317778983541328, 'recall'

In [ ]:
# result like table formate
results_df = pd.DataFrame(results).T
print(results_df)

                                                 classification_report  \
SVM                  {'0': {'precision': 0.7735849056603774, 'recal...   
Logistic Regression  {'0': {'precision': 0.8144329896907216, 'recal...   
Random Forest        {'0': {'precision': 0.7938144329896907, 'recal...   
XGBoost              {'0': {'precision': 0.8181818181818182, 'recal...   

                     accuracy  f1_score   roc_auc      confusion_matrix  
SVM                  0.733766  0.601942  0.805051  [[82, 17], [24, 31]]  
Logistic Regression  0.753247  0.660714  0.814692  [[79, 20], [18, 37]]  
Random Forest        0.727273     0.625  0.821488  [[77, 22], [20, 35]]  
XGBoost              0.720779  0.644628  0.776125  [[72, 27], [16, 39]]  


In [15]:
model.predict([[0,137, 40, 35, 168, 43.1, 2.288, 33]])

array([1])

In [16]:
#save model
import pickle
with open('diabetes_model.pkl', 'wb') as f:
    pickle.dump(model, f)